In [ ]:
import os
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import zipfile


# unzip file
zip_path = "Fruit_Classification-main (1).zip"
extract_path = "/content/fruit_project"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

os.listdir(extract_path)



In [ ]:
train_dir = "/content/fruit_project/Fruit_Classification-main/DATA/Fruits_Dataset_Train"
test_dir = "/content/fruit_project/Fruit_Classification-main/DATA/Fruits_Dataset_Test"

In [ ]:
# count images per class
def count_images_per_class(data_dir):
    class_counts = {}
    for cls in os.listdir(data_dir):
        cls_path = os.path.join(data_dir, cls)
        if os.path.isdir(cls_path):
            class_counts[cls] = len(os.listdir(cls_path))
    return class_counts

train_counts = count_images_per_class(train_dir)
test_counts = count_images_per_class(test_dir)

print("Train counts:", train_counts)
print("Test counts:", test_counts)


In [ ]:
import pandas as pd

labels_path = "/content/fruit_project/Fruit_Classification-main/DATA/Labels_Train.csv"
labels_df = pd.read_csv(labels_path)

labels_df.head()


In [ ]:
import matplotlib.pyplot as plt

# convert count to DataFrame
train_df = pd.DataFrame(list(train_counts.items()), columns=["Class", "Train Count"])
test_df = pd.DataFrame(list(test_counts.items()), columns=["Class", "Test Count"])

# merge
class_summary = pd.merge(train_df, test_df, on="Class")
class_summary = class_summary.sort_values(by="Class")

# graph
class_summary.set_index("Class")[["Train Count", "Test Count"]].plot(kind="bar", figsize=(10, 6))
plt.title("Image Count per Class (Train vs Test)")
plt.ylabel("Number of Images")
plt.xlabel("Class")
plt.xticks(rotation=0)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image

# look at image sizes
def get_image_sizes(data_dir, sample_limit=300):
    sizes = []
    count = 0
    for cls in os.listdir(data_dir):
        cls_path = os.path.join(data_dir, cls)
        if os.path.isdir(cls_path):
            for img_file in os.listdir(cls_path):
                img_path = os.path.join(cls_path, img_file)
                try:
                    with Image.open(img_path) as img:
                        sizes.append(img.size)  # (width, height)
                        count += 1
                        if count >= sample_limit:
                            return sizes
                except:
                    continue
    return sizes

# get sizes from training set
image_sizes = get_image_sizes(train_dir)

# graph
widths, heights = zip(*image_sizes)
plt.figure(figsize=(10, 5))
plt.hist(widths, bins=20, alpha=0.6, label='Widths')
plt.hist(heights, bins=20, alpha=0.6, label='Heights')
plt.title("Image Dimensions (Sample of 300)")
plt.xlabel("Pixels")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# summed each column to count images of each fruit
fruit_counts = labels_df.drop(columns=["FileName"]).sum().sort_values(ascending=False)
fruit_counts.plot(kind="bar", figsize=(12, 5), title="Image Count per Fruit")
plt.ylabel("Number of Images")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

In [ ]:
apple_df = labels_df[labels_df["Apple"] == 1]
print(f"Number of Apple images: {len(apple_df)}")
apple_df.head()

In [ ]:
def get_brightness(image):
    grayscale = image.convert("L")
    stat = grayscale.getextrema()
    return (stat[0] + stat[1]) / 2

brightness_scores = []
for cls in os.listdir(train_dir):
    cls_path = os.path.join(train_dir, cls)
    for img_name in random.sample(os.listdir(cls_path), 10):  # 10 per class
        with Image.open(os.path.join(cls_path, img_name)) as img:
            brightness_scores.append((cls, get_brightness(img)))

# convert to DataFrame
df_bright = pd.DataFrame(brightness_scores, columns=["Class", "Brightness"])
df_bright.groupby("Class")["Brightness"].plot(kind="kde", legend=True)
plt.title("Brightness Distribution per Class")
plt.show()